## Лабораторна робота №2. Частина 2
### *1. Здійснення Data Cleaning*
Відбувається:
- Приведення даних до робочого стану: заміна символів ? на стандартні значення NaN.
- Видалення порожніх рядків: очищення датасету від записів із відсутніми показниками.
- Об'єднання дати та часу: створення єдиного часового атрибута Datetime.
- Типізація даних: конвертація всіх фізичних величин у числовий формат float.
- Індексація: встановлення часової мітки як головного індексу для швидкого пошуку.

In [1]:
import pandas as pd
import numpy as np
import timeit

filename = 'power_data/household_power_consumption.txt'

def clean_data(file_path):
    df = pd.read_csv(file_path, sep=';', na_values=['?'], low_memory=False)
    
    # 2. Видалення пропущених значень (NaN)
    df.dropna(inplace=True)
    
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
    df.drop(['Date', 'Time'], axis=1, inplace=True)
    
    numeric_cols = df.columns.drop('Datetime')
    df[numeric_cols] = df[numeric_cols].astype(float)
    
    df.set_index('Datetime', inplace=True)
    
    return df

execution_time = timeit.timeit(lambda: clean_data(filename), number=1)

df_clean = clean_data(filename)

print(f"Очищення завершено за {execution_time:.2f} секунд.")
print(df_clean.info())
print("\nПерші 5 рядків очищених даних:")
print(df_clean.head())

Очищення завершено за 6.62 секунд.
<class 'pandas.DataFrame'>
DatetimeIndex: 2049280 entries, 2006-12-16 17:24:00 to 2010-11-26 21:02:00
Data columns (total 7 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Global_active_power    float64
 1   Global_reactive_power  float64
 2   Voltage                float64
 3   Global_intensity       float64
 4   Sub_metering_1         float64
 5   Sub_metering_2         float64
 6   Sub_metering_3         float64
dtypes: float64(7)
memory usage: 125.1 MB
None

Перші 5 рядків очищених даних:
                     Global_active_power  Global_reactive_power  Voltage  \
Datetime                                                                   
2006-12-16 17:24:00                4.216                  0.418   234.84   
2006-12-16 17:25:00                5.360                  0.436   233.63   
2006-12-16 17:26:00                5.374                  0.498   233.29   
2006-12-16 17:27:00                5.388        

### *2. Фільтрація за активною потужністю*
Відбувається:
- Відбір записів: пошук рядків, де показник Global_active_power перевищує 5 кВт.
- Виявлення піків: ідентифікація періодів максимального енергоспоживання.
- Профілювання: розрахунок середнього часу виконання фільтрації для оцінки продуктивності.

In [2]:
def filter_high_active_power(df):
    return df[df['Global_active_power'] > 5.0]

high_power_records = filter_high_active_power(df_clean)

time_taken = timeit.timeit(lambda: filter_high_active_power(df_clean), number=10) / 10

print(f"Знайдено записів: {len(high_power_records)}")
print(f"Середній час фільтрації: {time_taken:.5f} секунд")
print("\nПерші 5 записів вибірки:")
print(high_power_records.head())

Знайдено записів: 17547
Середній час фільтрації: 0.00414 секунд

Перші 5 записів вибірки:
                     Global_active_power  Global_reactive_power  Voltage  \
Datetime                                                                   
2006-12-16 17:25:00                5.360                  0.436   233.63   
2006-12-16 17:26:00                5.374                  0.498   233.29   
2006-12-16 17:27:00                5.388                  0.502   233.74   
2006-12-16 17:35:00                5.412                  0.470   232.78   
2006-12-16 17:36:00                5.224                  0.478   232.99   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
Datetime                                                                
2006-12-16 17:25:00              23.0             0.0             1.0   
2006-12-16 17:26:00              23.0             0.0             2.0   
2006-12-16 17:27:00              23.0             0.0             1.0   
2006-12-16 1

### *3. Складна фільтрація за напругою та струмом*
Відбувається:
- Багатофакторний аналіз: відбір записів із напругою понад 235 В та силою струму в межах 19-20 А.
- Порівняння груп: перевірка умови, за якої споживання пральної машини (Group 2) перевищує споживання бойлера (Group 3).
- Перевірка логіки: підтвердження коректності роботи логічних операторів у складних запитах.

In [3]:
import timeit

def filter_by_intensity_and_appliances(df):

    condition = (df['Global_intensity'].between(19, 20)) & (df['Sub_metering_2'] > df['Sub_metering_3'])
    return df[condition]

filtered_appliances_df = filter_by_intensity_and_appliances(df_clean)

timer = timeit.Timer(lambda: filter_by_intensity_and_appliances(df_clean))
avg_time = timer.timeit(number=100) / 100

print(f"Знайдено записів: {len(filtered_appliances_df)}")
print(f"Середній час виконання: {avg_time:.6f} секунд")
print("\nРезультати вибірки (перші 5 рядків):")
print(filtered_appliances_df[['Global_intensity', 'Sub_metering_2', 'Sub_metering_3']].head())

Знайдено записів: 2509
Середній час виконання: 0.007843 секунд

Результати вибірки (перші 5 рядків):
                     Global_intensity  Sub_metering_2  Sub_metering_3
Datetime                                                             
2006-12-16 18:09:00              19.0            37.0            16.0
2006-12-17 01:04:00              19.6            13.0             0.0
2006-12-17 01:08:00              19.6            27.0             0.0
2006-12-17 01:19:00              19.4            36.0             0.0
2006-12-17 01:20:00              19.4            35.0             0.0


### *4. Випадкова вибірка та статистичне оцінювання*
Відбувається:
- Статистичне моделювання: обрання 500,000 випадкових записів без повторів за допомогою методу sample.
- Агрегація даних: обчислення середніх показників для трьох основних груп енергоспоживання (кухня, пральня, бойлер).
- Аналіз продуктивності: проведення багаторазових тестів для визначення точного середнього часу виконання операції.
- Оцінка репрезентативності: отримання узагальненої статистики по всьому масиву даних через випадкову вибірку.

In [4]:
import timeit

def calculate_sample_averages(df, sample_size=500000):
    sample_df = df.sample(n=sample_size, replace=False)
    
    averages = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    
    return averages

result_averages = calculate_sample_averages(df_clean)

timer = timeit.Timer(lambda: calculate_sample_averages(df_clean))

avg_time = timer.timeit(number=10) / 10

print(f"--- Результати для випадкової вибірки (500,000 записів) ---")
print(f"Середній час виконання операції: {avg_time:.4f} секунд")
print("\nСередні значення споживання (Вт-год):")
print(result_averages)

--- Результати для випадкової вибірки (500,000 записів) ---
Середній час виконання операції: 0.1096 секунд

Середні значення споживання (Вт-год):
Sub_metering_1    1.136474
Sub_metering_2    1.297234
Sub_metering_3    6.459802
dtype: float64


### *5. Комбінована фільтрація та сегментація даних*
Відбувається:
- Складна фільтрація: відбір вечірніх записів (після 18:00) з екстремальним навантаженням понад 6 кВт.
- Порівняльний аналіз: виявлення періодів, коли споживання 2-ї групи приладів (пральня, холодильник) є домінуючим.
- Структурний поділ: розбиття отриманої вибірки на дві рівні частини для подальшої обробки.
- Семплювання за кроком: застосування специфічних зрізів — вибір кожного 3-го запису з першої половини та кожного 4-го з другої.
- Конкатенація: об'єднання оброблених сегментів у фінальний результуючий набір даних.

In [5]:
import pandas as pd
import timeit

def final_complex_task(df):
    condition = (
        (df.index.hour >= 18) & 
        (df['Global_active_power'] > 6.0) & 
        (df['Sub_metering_2'] > df['Sub_metering_1']) & 
        (df['Sub_metering_2'] > df['Sub_metering_3'])
    )
    
    filtered_df = df[condition]
    
    n_total = len(filtered_df)
    if n_total == 0:
        return filtered_df
    
    mid_point = n_total // 2
    first_half = filtered_df.iloc[:mid_point]
    second_half = filtered_df.iloc[mid_point:]
    
    res_part1 = first_half.iloc[::3]
    res_part2 = second_half.iloc[::4]
    
    final_result = pd.concat([res_part1, res_part2])
    
    return final_result

execution_time = timeit.timeit(lambda: final_complex_task(df_clean), number=1)
result_df = final_complex_task(df_clean)

print(f"--- Результати фінального завдання ---")
print(f"Час виконання: {execution_time:.4f} секунд")
print(f"Знайдено записів після фільтрації: {len(df_clean[(df_clean.index.hour >= 18) & (df_clean['Global_active_power'] > 6.0) & (df_clean['Sub_metering_2'] > df_clean['Sub_metering_1']) & (df_clean['Sub_metering_2'] > df_clean['Sub_metering_3'])])}")
print(f"Кількість записів після фінальних зрізів: {len(result_df)}")
print("\nПерші 10 записів результату:")
print(result_df.head(10))

--- Результати фінального завдання ---
Час виконання: 0.0543 секунд
Знайдено записів після фільтрації: 1061
Кількість записів після фінальних зрізів: 310

Перші 10 записів результату:
                     Global_active_power  Global_reactive_power  Voltage  \
Datetime                                                                   
2006-12-16 18:05:00                6.052                  0.192   232.93   
2006-12-16 18:08:00                6.308                  0.116   232.25   
2006-12-28 20:58:00                6.386                  0.374   236.63   
2006-12-28 21:02:00                8.088                  0.262   235.50   
2006-12-28 21:05:00                7.230                  0.152   235.22   
2006-12-28 21:08:00                7.352                  0.000   235.45   
2006-12-28 21:11:00                9.048                  0.000   231.48   
2006-12-28 21:14:00                9.118                  0.108   231.18   
2006-12-28 21:17:00                7.040                

### *6. Масштабування та стандартизація ознак*
Відбувається:
- Нормування (Min-Max Scaling): приведення всіх числових показників до єдиного діапазону від 0 до 1.
- Стандартизація (Z-score Normalization): трансформація даних так, щоб середнє значення дорівнювало 0, а стандартне відхилення — 1.
- Усунення розбіжностей: вирівнювання масштабів різних фізичних величин (вольти, ампери, вати) для коректного порівняння.
- Підготовка до аналізу: створення двох варіантів датасету для подальшого статистичного оброблення.
- Оцінка швидкодії: вимірювання часу виконання математичних перетворень над усім масивом даних.

In [6]:
import pandas as pd
import timeit

def scale_data(df):
    numeric_df = df.copy()
    
    df_norm = (numeric_df - numeric_df.min()) / (numeric_df.max() - numeric_df.min())
    
    df_std = (numeric_df - numeric_df.mean()) / numeric_df.std()
    
    return df_norm, df_std

execution_time = timeit.timeit(lambda: scale_data(df_clean), number=1)

df_normalized, df_standardized = scale_data(df_clean)

print(f"Масштабування завершено за {execution_time:.4f} секунд.")

print("\n--- Результат НОРМУВАННЯ (Min-Max 0-1) ---")
print(df_normalized.head(3))

print("\n--- Результат СТАНДАРТИЗАЦІЇ (Mean=0, Std=1) ---")
print(df_standardized.head(3))

Масштабування завершено за 0.5636 секунд.

--- Результат НОРМУВАННЯ (Min-Max 0-1) ---
                     Global_active_power  Global_reactive_power   Voltage  \
Datetime                                                                    
2006-12-16 17:24:00             0.374796               0.300719  0.376090   
2006-12-16 17:25:00             0.478363               0.313669  0.336995   
2006-12-16 17:26:00             0.479631               0.358273  0.326010   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
Datetime                                                                
2006-12-16 17:24:00          0.377593             0.0          0.0125   
2006-12-16 17:25:00          0.473029             0.0          0.0125   
2006-12-16 17:26:00          0.473029             0.0          0.0250   

                     Sub_metering_3  
Datetime                             
2006-12-16 17:24:00        0.548387  
2006-12-16 17:25:00        0.516129  
2006-12-16

### *7. Кореляційний аналіз показників*
Відбувається:
- Встановлення залежностей: розрахунок коефіцієнта Пірсона для оцінки сили лінійного зв'язку між потужністю та силою струму.
- Рангове оцінювання: обчислення коефіцієнта Спірмена для перевірки монотонності взаємозв'язку між атрибутами.
- Інтеграція SciPy: використання спеціалізованих математичних модулів для глибинного статистичного аналізу.
- Порівняння метрик: аналіз подібності результатів обох методів для підтвердження високої щільності зв'язку.
- Оцінка обчислювальної складності: вимірювання часу, необхідного для ранжування понад 2 мільйонів записів.

In [7]:
import timeit

def calculate_correlations(df, col1='Global_active_power', col2='Global_intensity'):
    pearson_corr = df[col1].corr(df[col2], method='pearson')
    
    spearman_corr = df[col1].corr(df[col2], method='spearman')
    
    return pearson_corr, spearman_corr

execution_time = timeit.timeit(lambda: calculate_correlations(df_clean), number=1)

pearson, spearman = calculate_correlations(df_clean)

print(f"Розрахунок завершено за {execution_time:.4f} секунд.")
print(f"Коефіцієнт Пірсона: {pearson:.5f}")
print(f"Коефіцієнт Спірмена: {spearman:.5f}")

Розрахунок завершено за 1.1790 секунд.
Коефіцієнт Пірсона: 0.99889
Коефіцієнт Спірмена: 0.99537


### *8. Категоризація та бінарне кодування (One Hot Encoding)*
Відбувається:
- Створення ознак (Feature Engineering): генерація нового категоріального атрибута TimeOfDay на основі годин споживання.
- Сегментація добових циклів: розподіл усіх записів на чотири часові проміжки: ранок, день, вечір та ніч.
- Трансформація типів: перетворення текстових категорій у набір бінарних стовпців (0 або 1) за допомогою методу get_dummies.
- Підготовка для машинного навчання: приведення даних до формату, що дозволяє алгоритмам враховувати часовий фактор без надання числової ваги годинам.
- Оцінка структурних змін: перевірка розширення розмірності датафрейму після додавання нових незалежних ознак.

In [8]:
import pandas as pd
import timeit

def perform_one_hot_encoding(df):
    def get_time_category(hour):
        if 6 <= hour < 12: return 'Morning'
        elif 12 <= hour < 18: return 'Afternoon'
        elif 18 <= hour < 23: return 'Evening'
        else: return 'Night'

    df_with_cat = df.copy()
    df_with_cat['TimeOfDay'] = df_with_cat.index.hour.map(get_time_category)
    
    df_encoded = pd.get_dummies(df_with_cat, columns=['TimeOfDay'], prefix='Time')
    
    return df_encoded

execution_time = timeit.timeit(lambda: perform_one_hot_encoding(df_clean), number=1)

df_final = perform_one_hot_encoding(df_clean)

print(f"One Hot Encoding завершено за {execution_time:.4f} секунд.")
print(f"Нові стовпці: {[col for col in df_final.columns if 'Time_' in col]}")
print("\nФрагмент даних з новими атрибутами:")
print(df_final[['Global_active_power', 'Time_Morning', 'Time_Afternoon', 'Time_Evening', 'Time_Night']].head())

One Hot Encoding завершено за 0.6314 секунд.
Нові стовпці: ['Time_Afternoon', 'Time_Evening', 'Time_Morning', 'Time_Night']

Фрагмент даних з новими атрибутами:
                     Global_active_power  Time_Morning  Time_Afternoon  \
Datetime                                                                 
2006-12-16 17:24:00                4.216         False            True   
2006-12-16 17:25:00                5.360         False            True   
2006-12-16 17:26:00                5.374         False            True   
2006-12-16 17:27:00                5.388         False            True   
2006-12-16 17:28:00                3.666         False            True   

                     Time_Evening  Time_Night  
Datetime                                       
2006-12-16 17:24:00         False       False  
2006-12-16 17:25:00         False       False  
2006-12-16 17:26:00         False       False  
2006-12-16 17:27:00         False       False  
2006-12-16 17:28:00         Fals